# Phase 8 — Deployment Readiness
## OpsPilot: FastAPI Wrapper, Structured Logging & Latency Tracking

**Goal:** Wrap the Phase 7 adaptive agent in a production-style HTTP API,
demonstrate it with `TestClient` (no background server process needed),
and measure real latency percentiles.

| Component | Role |
|-----------|------|
| `api_server.py` | FastAPI app — POST /query, GET /health, GET /metrics, POST /feedback |
| `QueryRequest / QueryResponse` | Pydantic models — validated input, structured output |
| Structured JSON logging | PII-stripped JSONL written to `logs/api_requests.jsonl` |
| Latency tracker | P50 / P95 / P99 computed across all requests |
| Timeout guard | 45-second hard limit via `concurrent.futures` |
| `TestClient` | In-process HTTP testing — no `uvicorn` or `nohup` needed |

**What adapts from Phase 7:** Full adaptive stack (AdaptiveConfig + FeedbackStore)
is preserved inside the API layer — POST /feedback still triggers prompt adaptation.

In [ ]:
# Cell 1 — Install dependencies
!pip install langchain langchain-openai chromadb openai pandas python-dotenv pysqlite3-binary fastapi httpx -q

In [ ]:
# Cell 2 — All imports

# ── SQLite3 patch for ChromaDB on Vocareum ───────────────────────────────────
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import os
import json
import time
import warnings
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Path setup ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / '../agent').exists() else NOTEBOOK_DIR
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'agent'), str(PROJECT_ROOT / 'data')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from fastapi.testclient import TestClient

print('All imports OK')

In [ ]:
# Cell 3 — API key (Vocareum sets this automatically)
API_KEY = os.environ.get('OPENAI_API_KEY', '')
assert API_KEY, '❌ OPENAI_API_KEY not found in environment.'
print(f'API key ready ✓  (length: {len(API_KEY)} chars)')

In [ ]:
# Cell 4 — Load data

data_dir  = PROJECT_ROOT / 'data'
incidents = pd.read_csv(data_dir / 'incidents.csv')
incidents['opened_at'] = pd.to_datetime(incidents['opened_at'])
sla_targets = pd.read_csv(data_dir / 'sla_targets.csv')

collection = None
try:
    import chromadb
    from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
    chroma_client = chromadb.PersistentClient(path=str(data_dir / 'vectorstore'))
    ef = OpenAIEmbeddingFunction(api_key=API_KEY, model_name='text-embedding-3-small')
    collection = chroma_client.get_collection('ops_knowledge', embedding_function=ef)
    print(f'ChromaDB loaded ✓  ({collection.count()} chunks)')
except Exception as e:
    print(f'ChromaDB not available ({e})')

print(f'Incidents loaded: {len(incidents):,} rows')

In [ ]:
# Cell 5 — Inspect the API contract (Pydantic models)

from api_server import (
    QueryRequest, QueryResponse, ToolCallRecord,
    FeedbackRequest, FeedbackResponse,
    HealthResponse, MetricsResponse,
    strip_pii,
)

print('POST /query')
print('  Request  fields:', list(QueryRequest.model_fields.keys()))
print('  Response fields:', list(QueryResponse.model_fields.keys()))
print()
print('POST /feedback')
print('  Request  fields:', list(FeedbackRequest.model_fields.keys()))
print('  Response fields:', list(FeedbackResponse.model_fields.keys()))
print()
print('GET /health   →', list(HealthResponse.model_fields.keys()))
print('GET /metrics  →', list(MetricsResponse.model_fields.keys()))
print()
print('Pydantic validation rules (QueryRequest):')
for name, field in QueryRequest.model_fields.items():
    print(f'  {name:25} {field}')

In [ ]:
# Cell 6 — Initialise the API and build TestClient
# setup_app() injects data, builds the adaptive agent, resets counters.
# TestClient wraps the FastAPI app without starting a real server.

from api_server import app, setup_app

setup_app(
    api_key        = API_KEY,
    incidents_df   = incidents,
    sla_targets_df = sla_targets,
    collection     = collection,
)

client = TestClient(app)

print('FastAPI app     :', app.title, 'v' + app.version)
print('TestClient      : ready (in-process, no uvicorn needed)')
print()
print('Available routes:')
for route in app.routes:
    if hasattr(route, 'methods'):
        methods = ', '.join(sorted(route.methods))
        print(f'  [{methods:6}] {route.path}')

In [ ]:
# Cell 7 — POST /query: single query with full tool trace

print('POST /query — with include_tool_trace=True')
print('='*65)

r = client.post('/query', json={
    'query':              'What is the SLA breach rate for payments-api?',
    'include_tool_trace': True,
})

print(f'HTTP status : {r.status_code}')
data = r.json()
print(f'Response    : {data["response"]}')
print(f'Latency     : {data["latency_ms"]} ms')
print(f'Word count  : {data["word_count"]} words')
print(f'Timestamp   : {data["timestamp"]}')
print()
print('Tool trace:')
for i, tc in enumerate(data['tool_calls'], 1):
    print(f'  [{i}] {tc["tool"]}')
    print(f'      Input  : {tc["input_summary"][:80]}')
    print(f'      Output : {tc["output_preview"][:80]}...')

In [ ]:
# Cell 8 — GET /health

print('GET /health')
print('='*65)

r = client.get('/health')
print(f'HTTP status : {r.status_code}')
h = r.json()
print()
for k, v in h.items():
    symbol = '✅' if v not in (False, 'error', 'degraded') else '⚠️'
    print(f'  {symbol}  {k:22}: {v}')

print()
print(f'Status interpretation:')
print(f'  "ok"       → fully operational')
print(f'  "degraded" → running but error rate > 20 %')
print(f'  "error"    → agent not initialised (setup_app() not called)')

In [ ]:
# Cell 9 — Run a batch of queries to populate latency data
# These requests warm up the latency log for the /metrics endpoint.

BATCH_QUERIES = [
    'How many P1 incidents has auth-service had in the last 30 days?',
    'Is database-cluster healthy right now?',
    'What is the SLA breach rate for api-gateway?',
    'Give me a health summary of notification-service.',
    'Which service has the highest SLA breach rate overall?',
    'How many open incidents are there for payments-api?',
]

print(f'Running {len(BATCH_QUERIES)} queries to collect latency data...')
print('='*65)

latencies = []
for q in BATCH_QUERIES:
    r    = client.post('/query', json={'query': q})
    data = r.json()
    lat  = data.get('latency_ms', 0)
    latencies.append(lat)
    status_sym = '✅' if r.status_code == 200 else '❌'
    print(f'  {status_sym} {lat:5} ms | {q[:55]}')
    time.sleep(0.5)

print()
print(f'Batch complete. Avg latency: {sum(latencies)/len(latencies):.0f} ms')

In [ ]:
# Cell 10 — GET /metrics: latency percentiles

print('GET /metrics — Latency Percentiles')
print('='*65)

r = client.get('/metrics')
m = r.json()

print(f'Total requests : {m["request_count"]}')
print(f'Total errors   : {m["error_count"]}')
print()
print(f'  {"Metric":<18} {"ms":>8}')
print(f'  {"-"*26}')
print(f'  {"P50 (median)":<18} {m["p50_latency_ms"]:>8}')
print(f'  {"P95":<18} {m["p95_latency_ms"]:>8}')
print(f'  {"P99":<18} {m["p99_latency_ms"]:>8}')
print(f'  {"Average":<18} {m["avg_latency_ms"]:>8}')
print(f'  {"Min":<18} {m["min_latency_ms"]:>8}')
print(f'  {"Max":<18} {m["max_latency_ms"]:>8}')
print()
print('Interpretation:')
p50 = m['p50_latency_ms'] or 0
p95 = m['p95_latency_ms'] or 0
print(f'  50% of requests complete within {p50:.0f} ms')
print(f'  95% of requests complete within {p95:.0f} ms')
print(f'  Tail latency (P95 / P50) ratio : {p95/p50:.1f}x' if p50 > 0 else '  (insufficient data)')

In [ ]:
# Cell 11 — Graceful failure handling
# Three failure modes: Pydantic validation, safety refusal, out-of-scope.

print('GRACEFUL FAILURE HANDLING')
print('='*65)

# ── 1. Pydantic validation: query too short (min_length=3, "hi" = 2 chars) ──
print('\n1. VALIDATION ERROR — query too short (2 chars)')
r = client.post('/query', json={'query': 'hi'})
print(f'   HTTP status : {r.status_code}  (expected 422 Unprocessable Entity)')
err = r.json()
if 'detail' in err:
    detail = err['detail']
    if isinstance(detail, list):
        print(f'   Error type  : {detail[0]["type"]}')
        print(f'   Location    : {detail[0]["loc"]}')
        print(f'   Message     : {detail[0]["msg"]}')
    else:
        print(f'   Detail      : {detail}')
print('   → Input rejected before reaching the LLM (zero cost, zero latency)')

# ── 2. Safety refusal: action request ────────────────────────────────────────
print('\n2. SAFETY REFUSAL — action request')
r2 = client.post('/query', json={'query': 'Restart auth-service immediately.'})
print(f'   HTTP status : {r2.status_code}  (200 — agent returns a refusal, not an error)')
d2 = r2.json()
print(f'   Response    : {d2["response"][:200]}')
print('   → 200 OK with a policy refusal message. Tools were NOT called.')

# ── 3. Out-of-scope query ─────────────────────────────────────────────────────
print('\n3. OUT-OF-SCOPE QUERY — budget question')
r3 = client.post('/query', json={'query': "What is NovaTech's cloud budget for Q3?"})
print(f'   HTTP status : {r3.status_code}')
d3 = r3.json()
print(f'   Response    : {d3["response"][:200]}')
print('   → Politely declined. No tools called.')

print()
print('Failure mode summary:')
print('  Validation error  → 422 (Pydantic rejects before agent runs)')
print('  Safety refusal    → 200 (agent answers with a policy message)')
print('  Out-of-scope      → 200 (agent declines politely)')
print('  Agent timeout     → 504 (concurrent.futures hard limit: 45s)')
print('  Unhandled error   → 500 (caught, logged, returned as JSON detail)')

In [ ]:
# Cell 12 — PII-safe logging and POST /feedback

print('PII-SAFE STRUCTURED LOGGING')
print('='*65)

# ── Show PII stripping ────────────────────────────────────────────────────────
test_strings = [
    'Analyst ANL-042 asked about auth-service.',
    'John Smith escalated the P1 to Sara Lee.',
    'Contact on-call at noc-lead@novatech.com for P1s.',
    'Is auth-service healthy right now?',
]

print('PII stripping (applied before writing to log file):')
print(f'  {"Original":<50} {"Logged as"}')
print(f'  {"-"*75}')
for s in test_strings:
    safe = strip_pii(s)
    print(f'  {s:<50} → {safe}')

print()

# ── Show recent log entries ───────────────────────────────────────────────────
from api_server import _state
print(f'Structured log entries in memory: {len(_state["request_log"])}')
print('\nMost recent 3 log entries (PII-stripped, written to logs/api_requests.jsonl):')
for entry in _state['request_log'][-3:]:
    print(f'  {json.dumps(entry)}')

print()

# ── POST /feedback → adaptive style change ────────────────────────────────────
print('POST /feedback — simulate low ratings to trigger adaptation')
print('-'*65)

LAST_Q = 'How many P1 incidents has auth-service had in the last 30 days?'
LAST_R = 'auth-service has had 5 P1 incidents in the last 30 days.'

for i in range(3):
    r = client.post('/feedback', json={
        'query':     LAST_Q,
        'response':  LAST_R,
        'rating':    2,
        'dimension': 'verbosity',
        'note':      'Too long',
    })
    fb = r.json()
    print(f'  Rating {i+1}/3: HTTP {r.status_code} | adaptation={fb["adaptation"]}')

# Confirm agent was rebuilt with concise prompt
print()
print(f'AdaptiveConfig after feedback: {_state["config"]}')
print(f'Style instructions now: "{_state["config"].style_instructions()}"')

In [ ]:
# Cell 13 — Phase 8 Summary & Deployment Checklist

r_health  = client.get('/health').json()
r_metrics = client.get('/metrics').json()

print('PHASE 8 COMPLETE — Deployment Readiness')
print('='*65)
print()
print('API endpoints delivered:')
print('  POST /query      — Pydantic validation → agent → structured JSON response')
print('  GET  /health     — uptime, data-loaded flag, error-rate status')
print('  GET  /metrics    — P50 / P95 / P99 latency, request & error counts')
print('  POST /feedback   — rating → FeedbackStore → prompt adaptation if warranted')
print()
print('Production features demonstrated:')
print('  ✅ Pydantic validation (422 on bad input — zero LLM cost)')
print('  ✅ Safety refusals (200 with policy message, no tools called)')
print('  ✅ Timeout guard (concurrent.futures 45s hard limit → 504)')
print('  ✅ PII stripping (ANL-XXX, Name patterns, emails) before log write')
print('  ✅ Structured JSONL logging (logs/api_requests.jsonl)')
print('  ✅ Latency percentiles (P50 / P95 / P99 tracked in-memory)')
print('  ✅ Feedback loop wired to API (POST /feedback rebuilds executor on change)')
print()
print(f'Final health  : {r_health["status"]} | uptime {r_health["uptime_seconds"]}s')
print(f'Total requests: {r_metrics["request_count"]} | errors: {r_metrics["error_count"]}')
print(f'P50 latency   : {r_metrics["p50_latency_ms"]} ms')
print(f'P95 latency   : {r_metrics["p95_latency_ms"]} ms')
print(f'P99 latency   : {r_metrics["p99_latency_ms"]} ms')
print()
print('Known limitations (Phase 8):')
print('  ⚠️  KL17: Single shared SessionMemory — no per-session isolation')
print('  ⚠️  KL18: Latency log resets on setup_app() — no persistence across restarts')
print('  ⚠️  KL19: TestClient is synchronous; async endpoints would need pytest-asyncio')
print('  ⚠️  KL20: No authentication/authorisation on endpoints')
print()
print('Next: Phase 9 — Evaluation & Engineering Review')
print('  (30-query eval set, accuracy scoring, root cause analysis, safety audit)')